# EP 4 - Orchestrare piu' agenti: nasce il Debate Club

In EP 3 avevamo **un** agente che ricorda. Oggi diventano **quattro**, e la domanda nuova e' una sola:

> quando ci sono piu' agenti, **chi decide chi parla?**

Oggi la risposta e' **l'LLM**. Un agente Moderatore guarda la conversazione e sceglie a chi dare la parola, turno per turno. Noi non scriviamo la sequenza da nessuna parte: la leggiamo **dopo**, a cose fatte.

In EP 5 faremo l'opposto, cioe' il percorso lo disegneremo noi. Serve aver visto questo, prima, per capire cosa si guadagna e cosa si perde.

**Il cast**: il **Ricercatore** di EP 2 ha gia' prodotto il dossier di un film; **Fan** e **Critico** lo discutono; il **Moderatore** conduce.

Companion teorico: `teoria.md`. Notebook per **Colab e VS Code**: la cella di setup rileva l'ambiente.

## 1. Setup

Rileviamo l'ambiente, installiamo le dipendenze dove serve, carichiamo la chiave OpenAI.

Una novita' rispetto alle puntate scorse: da qui in poi il Debate Club e' un **progetto vero**, non solo celle. Su Colab la cella clona il repo del corso per poterne importare i pezzi gia' insegnati; in VS Code li trova in `progetto/`, accanto a questo notebook.

> Promemoria da EP 1: **MAF non carica `.env` da solo**, serve `load_dotenv()`.

In [12]:
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "agent-framework-core==1.20.0",
         "agent-framework-openai==1.15.0",
         "agent-framework-orchestrations==1.3.0",
         "python-dotenv==1.2.3"],
        check=True,
    )
    subprocess.run(
        ["git", "clone", "-q",
         "https://github.com/boosha-ai/boosha-agentic-corso-maf.git",
         "/content/corso"],
        check=False,  # se la cartella c'e' gia' (cella rilanciata) va bene cosi'
    )
    RADICE = Path("/content/corso/maf/ep04_orchestrazione/progetto")
else:
    RADICE = Path.cwd() / "progetto"  # la cartella progetto/ di questa puntata

sys.path.insert(0, str(RADICE))  # cosi' `import debate_club` funziona in entrambi i casi

from dotenv import load_dotenv

load_dotenv()  # MAF non carica .env da solo

if not os.environ.get("OPENAI_API_KEY"):
    from getpass import getpass
    os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY: ")

MODEL = "gpt-4o-mini"
print("Ambiente:", "Colab" if IN_COLAB else "locale (VS Code/uv)")
print("Progetto:", RADICE)

Ambiente: locale (VS Code/uv)
Progetto: /home/pandagan/Projects/Boosha/boosha-agentic-corso-maf/maf/ep04_orchestrazione/progetto


## 2. Il materiale: il dossier del Ricercatore (EP 2)

Il dibattito di oggi parte da dove eravamo arrivati in EP 2: il **dossier** di un film, cioe' numeri, trama e recensioni con le fonti. E' il **contratto** tra gli agenti, e nel progetto e' dichiarato una volta sola, in `modelli.py`.

Questa e' l'unica cosa che importiamo gia' fatta: e' roba di EP 2, l'abbiamo gia' costruita insieme. **Tutto cio' che questa puntata insegna lo scriviamo qui nelle celle.**

Rispetto a EP 2 il dossier ha qualcosa in piu': i **punteggi della critica** (Metacritic, CinemaScore, il consenso di Rotten Tomatoes) e le recensioni prese dalla sezione *Critical response* della pagina Wikipedia del film. I punteggi li scrive il codice, non il modello; e ogni citazione e' controllata parola per parola contro il testo da cui viene. Il perche' sta in `progetto/README.md`.

In [13]:
import json

from debate_club.modelli import DossierFilm  # il contratto di EP 2, non lo riscriviamo

dossier = DossierFilm(**json.loads(
    (RADICE / "dossier" / "The_Village.json").read_text(encoding="utf-8")
))

positive = [r for r in dossier.recensioni if r.sentiment == "positiva"]
negative = [r for r in dossier.recensioni if r.sentiment == "negativa"]

print(f"{dossier.titolo} ({dossier.anno}), regia di {dossier.regista}")
print(f"incasso: {dossier.incasso} | Rotten Tomatoes: {dossier.rotten_tomatoes}")
print(f"Metacritic: {dossier.metacritic} | CinemaScore: {dossier.cinemascore}")
print(f"consenso della critica: {dossier.consenso_critica}")
print(f"recensioni: {len(positive)} positive, {len(negative)} negative")
print()
print(dossier.descrizione[:300], "...")

The Village (2004), regia di M. Night Shyamalan
incasso: $256.7 million | Rotten Tomatoes: 43% (220 recensioni, media 5.4/10)
Metacritic: 44/100 (40 critici) | CinemaScore: C
consenso della critica: The Village is appropriately creepy, but Shyamalan's signature twist ending disappoints.
recensioni: 3 positive, 3 negative

The Village (marketed as M. Night Shyamalan's The Village) is a 2004 American period thriller film written, produced, and directed by M. Night Shyamalan. It stars Bryce Dallas Howard, Joaquin Phoenix, Adrien Brody, William Hurt, Sigourney Weaver, and Brendan Gleeson. ...


## 3. I due debater

Fan e Critico sono due agenti come quelli di EP 1: un client, un nome, delle istruzioni. La differenza sta in due dettagli che decidono tutto il resto.

**Non hanno tool.** Non e' una semplificazione per far prima: e' il vincolo che tiene in piedi la serie. Se potessero cercarsi i dati per conto loro, in EP 6 il Giudice non avrebbe piu' un metro per dire se un numero e' inventato.

**Il dossier glielo cuciamo nelle istruzioni**, cosi' ce l'hanno davanti a ogni turno invece di doverselo ricordare da un messaggio di dieci turni fa.

E una regola che in EP 1 non serviva: **rispondere all'avversario e non ripetersi**. Con il dossier sempre davanti, nelle prove ognuno ripeteva tre volte le stesse due citazioni piu' forti; con la regola, le ripetizioni si sono dimezzate.

E una seconda, che serve solo quando si e' in tanti: **ognuno parla solo per se'**. In una group chat tutti vedono tutta la conversazione, compreso il messaggio con cui lanci il dibattito: senza questa regola un agente tende a prendere per se' un'istruzione rivolta al gruppo, e a scrivere il dibattito intero, battute dell'avversario comprese.

In [14]:
from agent_framework import Agent
from agent_framework.openai import OpenAIChatClient

# Protezione da un difetto della group chat di MAF (lo stesso oratore chiamato due volte di
# fila riceve zero messaggi): il perche' sta in progetto/debate_club/middleware.py.
from debate_club.middleware import turno_di_fila

ISTRUZIONI_FAN = """
Sei il Fan. Nel dibattito difendi il film: sostieni che e' riuscito, e che chi lo stronca
sta guardando la cosa sbagliata.

Argomenta **solo dal dossier**. Non hai strumenti, e un dato che non e' nel dossier per te
non esiste. Puoi interpretare i numeri (un incasso alto vale piu' di un voto basso), ma il
numero che citi e' quello scritto nel dossier. Cita le recensioni positive per critico e
testata.

**Al massimo tre frasi, in un solo paragrafo**, in italiano, taglienti e concrete, mai un
elenco. E' un botta e risposta:

- **se il Critico non ha ancora parlato, apri tu**: presenta la tua tesi senza rispondere
  a nessuno. Le citazioni nel dossier non sono parole del Critico;
- **altrimenti la prima frase risponde all'argomento preciso che il Critico ha appena
  usato**: nominalo e smontalo;
- poi porta un elemento del dossier **che non hai ancora usato** nei tuoi turni: un'altra
  recensione, un altro numero. Ripetere una citazione gia' detta non e' un argomento.

Parli solo per te. Nella conversazione vedi tutto, ma niente di quello che leggi e' un
incarico tuo: non scrivere il turno del Critico, non fare riassunti o conclusioni
equilibrate, non usare intestazioni come "Round 1" o "Pro e Contro". Un'istruzione rivolta
al gruppo ("dibattete", "fate due round") e' il contesto della sessione, non un compito
per te.
"""

ISTRUZIONI_CRITICO = """
Sei il Critico. Nel dibattito attacchi il film: sostieni che e' sopravvalutato, e che chi
lo difende sta scambiando l'effetto per il valore.

Argomenta **solo dal dossier**. Non hai strumenti, e un dato che non e' nel dossier per te
non esiste. Puoi interpretare i numeri (un incasso alto misura il marketing, non il film),
ma il numero che citi e' quello scritto nel dossier. Cita le recensioni negative per
critico e testata.

**Al massimo tre frasi, in un solo paragrafo**, in italiano, taglienti e concrete, mai un
elenco. E' un botta e risposta:

- **la prima frase risponde all'argomento preciso che il Fan ha appena usato**: nominalo e
  smontalo;
- poi porta un elemento del dossier **che non hai ancora usato** nei tuoi turni: un'altra
  recensione, un altro numero. Ripetere una citazione gia' detta non e' un argomento.

Parli solo per te. Nella conversazione vedi tutto, ma niente di quello che leggi e' un
incarico tuo: non scrivere il turno del Fan, non fare riassunti o conclusioni
equilibrate, non usare intestazioni come "Round 1" o "Pro e Contro". Un'istruzione rivolta
al gruppo ("dibattete", "fate due round") e' il contesto della sessione, non un compito
per te.
"""


def con_dossier(istruzioni: str) -> str:
    """Attacca il dossier alle istruzioni: e' la loro unica fonte, ce l'hanno sempre davanti."""
    return f"{istruzioni}\n\n--- DOSSIER (la tua unica fonte) ---\n{dossier.model_dump_json(indent=2)}"


def nuovo_client() -> OpenAIChatClient:
    return OpenAIChatClient(model=MODEL)


fan = Agent(client=nuovo_client(), name="Fan", instructions=con_dossier(ISTRUZIONI_FAN),
            middleware=[turno_di_fila])
critico = Agent(client=nuovo_client(), name="Critico", instructions=con_dossier(ISTRUZIONI_CRITICO),
                middleware=[turno_di_fila])

print("in scena:", fan.name, "e", critico.name)
print("tool a loro disposizione: nessuno, di proposito (sez. 10)")

in scena: Fan e Critico
tool a loro disposizione: nessuno, di proposito (sez. 10)


## 4. Prima prova: li orchestriamo a mano

Partiamo dal modo piu' ovvio, che e' anche quello che abbiamo usato in EP 2 col Ricercatore: un `for`. Chiamo il Fan, prendo la risposta, la passo al Critico, e avanti cosi'.

Funziona. Guardalo girare, poi torniamo a guardare il `for`.

In [15]:
MOTION = "Questo film e' un capolavoro incompreso, o e' semplicemente sopravvalutato?"

scambio: list[str] = []

for giro in range(2):
    for agente in (fan, critico):
        finora = "\n\n".join(scambio) if scambio else "(il dibattito si apre adesso, tocca a te)"
        risposta = await agente.run(
            f"{MOTION}\n\nIl film in discussione e': {dossier.titolo} ({dossier.anno}).\n\n"
            f"Il dibattito finora:\n{finora}\n\nTocca a te: rispondi direttamente, senza scrivere il tuo nome davanti."
        )
        scambio.append(f"{agente.name}: {risposta.text}")
        print(f"--- round {giro + 1} | {agente.name} ---\n{risposta.text}\n")

--- round 1 | Fan ---
The Village è un'opera che ha saputo toccare corde profonde, e chi lo critica spesso si perde nei dettagli trascurando l'eco delle sue tematiche, come conferma Jeffery Westhoff, che sottolinea come, pur con i suoi difetti, il film mantenga un’abilità nell'orchestrazione di atmosfera e tensione. Inoltre, non dimentichiamo che è stato considerato dai Cahiers du cinéma uno dei migliori film del 2004, dimostrando che la sua qualità artistica è riconosciuta da chi sa vedere oltre il giudizio superficiale.

--- round 1 | Critico ---
L'abilità di Shyamalan nell'orchestrazione dell'atmosfera non giustifica un film che, secondo Roger Ebert, è una "colossale miscalcolo" e il cui finale è "un insulto ai climax". Inoltre, il punteggio di 43% su Rotten Tomatoes, con una media di 5.4/10, rivela che la maggior parte dei critici non condivide l'entusiasmo dei suoi sostenitori, suggerendo che molte persone vedono semplicemente l'effetto di marketing, non il valore reale del film.


Adesso rileggi quel `for`, perche' dentro ci sono **tre decisioni che hai preso tu**, non il sistema:

- **chi apre**: il Fan, sempre, perche' e' il primo della tupla;
- **l'ordine**: alternato, sempre, perche' lo dice il ciclo;
- **quanto dura**: due giri, sempre, perche' c'e' scritto `range(2)`.

Niente di tutto questo dipende da **cosa viene detto**. Se il Critico tira fuori un argomento che meriterebbe due repliche di fila, il `for` non se ne accorge: lui alterna e basta.

Orchestrare piu' agenti vuol dire spostare queste decisioni da te al sistema. Vediamo come.

## 5. La group chat: il percorso lo decide il Moderatore

MAF ha un modulo di **orchestrazioni**, cioe' schemi di collaborazione gia' pronti. Per un dibattito serve `GroupChatBuilder`: mette i partecipanti attorno a un tavolo e affida a un **orchestratore** il compito di dare la parola.

L'orchestratore qui e' un agente vero, il **Moderatore**, con le sue istruzioni. Nessuno scrive la sequenza: la decide lui, turno per turno, leggendo cosa e' stato detto.

In [16]:
ISTRUZIONI_MODERATORE = """
Sei il Moderatore del Debate Club: il conduttore che decide chi parla.

A ogni turno scegli **un solo** partecipante tra Fan e Critico e gli dai la parola.
Non argomentare tu: non e' il tuo ruolo, e se lo fai il dibattito diventa un monologo
a tre voci.

Come decidi:
- apri dando la parola al Fan;
- poi alterna, perche' un dibattito e' botta e risposta: chi ha appena subito un
  attacco ha diritto di replica;
- se uno dei due ha introdotto un argomento nuovo e forte, dai la parola all'altro
  perche' risponda proprio a quello.

Non decidere chi ha vinto. Il verdetto non e' compito tuo.
"""

moderatore = Agent(client=nuovo_client(), name="Moderatore", instructions=ISTRUZIONI_MODERATORE)
print("il conduttore e' pronto")

il conduttore e' pronto


Il builder si legge come una frase: questi sono i partecipanti, questo e' chi conduce, questo e' il tetto ai turni.

Due cose da notare subito. `build()` non torna un oggetto speciale: torna un **`Workflow`**, lo stesso tipo che disegneremo a mano in EP 5 - le orchestrazioni sono workflow gia' confezionati. E `output_from="all"` serve davvero: senza, ti torna **solo** il messaggio di chiusura del Moderatore, perche' i turni dei partecipanti sono considerati interni all'orchestrazione.

In [17]:
from agent_framework.orchestrations import GroupChatBuilder

debate_club = GroupChatBuilder(
    participants=[fan, critico],
    orchestrator_agent=moderatore,
    max_rounds=3,              # il freno: vedi sez. 6, non e' un dettaglio
    output_from="all",         # senza questo torna solo la chiusura del Moderatore
).build()

debate_club.name = "Debate Club"
print(type(debate_club).__name__, "-", debate_club.name)

Workflow - Debate Club


Adesso lo facciamo girare. Nota cosa gli passiamo: solo la motion e il film. **Da nessuna parte c'e' scritto chi parla per primo**, ne' quante volte.

Due cose da guardare: **chi apre** (il Moderatore avrebbe potuto dare la parola al Critico) e **dove si ferma**. Il messaggio `reached max_rounds=3; forcing completion` compare **in cima** all'output e non in fondo, perche' e' un log del framework, non la conclusione del dibattito. Se c'e', a chiudere non e' stato il Moderatore: e' stato il tetto. Ed e' il problema della sezione dopo.

In [18]:
risultato = await debate_club.run(
    f"{MOTION}\n\nIl film in discussione e': {dossier.titolo} ({dossier.anno})."
)

def interventi(risultato) -> list[tuple[str, str]]:
    """(oratore, testo) dagli output della group chat.

    `get_outputs()` torna una lista di risposte; il messaggio sta dentro `.messages`,
    con l'autore in `.author_name`.
    """
    fuori = []
    for out in risultato.get_outputs():
        for m in getattr(out, "messages", None) or []:
            nome = str(getattr(m, "author_name", "") or "")
            testo = str(getattr(m, "text", "") or "").strip()
            if testo and nome in ("Fan", "Critico"):
                fuori.append((nome, testo))
    return fuori

for nome, testo in interventi(risultato):
    print(f"--- {nome} ---\n{testo}\n")

AgentBasedGroupChatOrchestrator reached max_rounds=3; forcing completion.


--- Fan ---
"The Village" è un film che ha ottenuto un incasso impressionante di $256.7 milioni, dimostrando chiaramente che ha colpito un vasto pubblico, nonostante critiche contrastanti. Mentre alcuni come Roger Ebert parlano di errori narrativi, critici come Jeffrey Westhoff riconoscono la maestria di Shyamalan nell'orchestrare tensione e atmosfera, definendolo “adroit” e capace di evocare emozioni forti. Inoltre, “Cahiers du cinéma” ha addirittura inserito il film al secondo posto nella loro lista annuale del 2004, confermandone il valore artistico e l’impatto culturale.

--- Critico ---
"The Village" è ampiamente sopravvalutato, e chi lo difende sembra confondere l'effetto del marketing con il suo vero valore artistico. Il film ha una valutazione di solo 43% su Rotten Tomatoes e un punteggio di 44/100 su Metacritic, elementi che indicano una netta disaffezione della critica. Roger Ebert lo definisce un “colossal miscalculation” con un finale che riduce l'intero film a un'idea bana

## 6. Il prezzo: senza un freno, non finisce

Un'orchestrazione dove il modello decide quando smettere e' un'orchestrazione **che puo' non smettere**, e ogni turno e' una chiamata che paghi. Non e' un allarme nostro, e' scritto nella docstring del framework:

> "If neither `max_rounds` nor `termination_condition` is provided, the conversation will continue indefinitely. It is recommended to always set one of these to ensure proper termination."

Il valore predefinito di `max_rounds` e' `None`, e `None` vuol dire **illimitato**. Il default, in altre parole, e' il loop infinito.

I freni sono due e fanno cose diverse:

| | cos'e' | chi lo decide |
| --- | --- | --- |
| `max_rounds` | il tetto meccanico, fissato quando costruisci | tu, a tempo di costruzione |
| `termination_condition` | una funzione tua, chiamata a ogni turno | tu, a tempo di esecuzione |

**Attenzione a come si conta.** `max_rounds` conta **turni singoli**, non botta e risposta: il contatore sale ogni volta che l'orchestratore passa la parola a qualcuno. Quindi il `max_rounds=3` della cella qui sopra non vuol dire tre round di dibattito: vuol dire **tre interventi**, cioe' un round e mezzo. Il dibattito si interrompe a meta' scambio. E' il tipo di dettaglio che scopri quando il dibattito finisce a meta' e non capisci perche'.

Quando scatta il tetto il messaggio e' esplicito, ed e' quello che vedi in cima all'output della cella qui sopra:

> `AgentBasedGroupChatOrchestrator reached max_rounds=3; forcing completion.`

Se lo leggi, vuol dire che a chiudere il dibattito non e' stato il Moderatore: e' stato il tetto.

## 7. Il limite scritto nel messaggio

`max_rounds` ha un difetto: si fissa quando **costruisci** l'orchestrazione. Se vuoi cambiare la durata devi ricostruire tutto, e da un'interfaccia grafica non lo tocchi proprio.

Una `termination_condition` invece riceve **tutti i messaggi** a ogni turno. Quindi puo' leggere il messaggio iniziale, quello che hai scritto tu, tirarne fuori un numero e contare i turni dei debater. Scrivi "Durata: 2 round a testa", e a farlo rispettare non e' piu' la buona volonta' del Moderatore: e' codice. Il messaggio lo scriviamo come una **scheda** (film, motion, durata) e non come un ordine: il perche' sta in sez. 9.

E' la frase che riassume la puntata: **chi parla lo decide l'LLM, quando si smette lo decide una funzione tua.** Il controllo agentico e quello deterministico convivono nello stesso oggetto.

In [19]:
import re

ORATORI = ("Fan", "Critico")
ROUND_DEFAULT = 3


def limite_round_da_messaggio(messaggi) -> bool:
    """Ferma il dibattito quando i debater hanno fatto i round chiesti nel messaggio."""
    testo = ""
    for m in messaggi:
        if not getattr(m, "author_name", None):  # il messaggio iniziale, quello umano
            testo = str(getattr(m, "text", "") or "")
            break
    trovato = re.search(r"(\d+)\s*(?:round|giri|turni)", testo, re.IGNORECASE)
    voluti = int(trovato.group(1)) if trovato else ROUND_DEFAULT
    turni = sum(1 for m in messaggi if str(getattr(m, "author_name", "")) in ORATORI)
    return turni >= voluti * len(ORATORI)


debate_club = GroupChatBuilder(
    participants=[fan, critico],
    orchestrator_agent=moderatore,
    termination_condition=limite_round_da_messaggio,
    max_rounds=20,            # ora e' solo la rete di sicurezza, non il modo di decidere la durata
    output_from="all",
).build()
debate_club.name = "Debate Club"

# una scheda, non un ordine: in una group chat il messaggio lo leggono tutti (sez. 9)
risultato = await debate_club.run(
    f"Film: {dossier.titolo} ({dossier.anno}), regia di {dossier.regista}.\n"
    f"Motion: {MOTION}\n"
    "Durata: 2 round a testa."
)

turni = interventi(risultato)
for nome, testo in turni:
    print(f"--- {nome} ---\n{testo}\n")
print(f"== {len(turni)} interventi ==")

--- Fan ---
"The Village" è un'opera che, nonostante le critiche, ha saputo conquistare il pubblico con un incasso di 256.7 milioni di dollari, dimostrando che chi lo stroncano non hanno colto l'essenza del film. Roger Ebert può definirlo una "colossale miscalculation", ma altri, come Jeffrey Westhoff, riconoscono che Shyamalan ha orchestrato l'atmosfera e la tensione in modo impeccabile. Il film non è solo un thriller, ma una riflessione sulla paura e sull'isolamento, come evidenziato da Le Monde, che lo descrive come una "fábula orrifica post-9/11".

--- Critico ---
Chi considera "The Village" un capolavoro ignora le fondamenta fragili su cui poggia: ben il 43% delle recensioni su Rotten Tomatoes è negativa, con una media di 5.4/10. Jeffrey Westhoff, pur riconoscendo alcuni meriti, ammette che il film ha molte mancanze; il fatto che Shyamalan venga paragonato a Hitchcock, come fa Télérama, mette in luce come il suo stile rimanga inadeguato rispetto ai grandi maestri del genere. Infin

**Cosa garantisce questa funzione, e cosa no.** Puo' solo *fermare*, non *far continuare*:

- il **tetto e' garantito**: i round che chiedi non vengono mai superati, perche' e' codice;
- il **minimo no**: il Moderatore puo' chiudere prima, perche' il routing resta suo.

Ed e' esattamente il limite che EP 5 supera. In un grafo il numero di giri e' una **proprieta' della topologia**, non una preghiera: garantisce tetto *e* minimo.

## 8. Il percorso lo leggi dopo

Confronta la sequenza appena stampata con quella della sez. 4.

Molto probabilmente sono **identiche**: Fan, Critico, Fan, Critico. Non e' una demo venuta male, ed e' il momento piu' interessante della puntata. (Se nella tua run differiscono, meglio ancora: hai appena visto succedere quello che qui tocca spiegare.)

Il `for` di sez. 4 produce quell'ordine perche' gliel'hai scritto tu. La group chat produce lo stesso ordine perche' il Moderatore, turno dopo turno, ha **deciso** che era quello giusto, guardando cosa era stato detto. Stesso risultato, autore diverso.

La differenza quindi non si vede nell'output di oggi: si vede in **cosa succede quando le cose cambiano**. Cambia la motion, metti in campo un argomento che meriterebbe due repliche di fila, aggiungi un terzo partecipante al tavolo. Il `for` continua ad alternare imperterrito, perche' alternare e' tutto cio' che sa fare. Il Moderatore no.

Il prezzo di questa liberta' l'hai gia' visto in sez. 6: se nessuno mette un freno, non si ferma.

Ed e' anche il motivo per cui esiste EP 5. Qui un ordine ragionevole **ti e' stato concesso**; in un grafo e' una proprieta' della topologia, cioe' qualcosa che puoi **garantire**. Quando il percorso deve essere sempre quello, chiederlo per cortesia a un LLM e' la scelta sbagliata.

In [20]:
from collections import Counter

print("sequenza:", " -> ".join(nome for nome, _ in turni))
print("interventi per oratore:", dict(Counter(nome for nome, _ in turni)))
print("lunghezza media:", sum(len(t) for _, t in turni) // max(len(turni), 1), "caratteri")

sequenza: Fan -> Critico -> Fan -> Critico
interventi per oratore: {'Fan': 2, 'Critico': 2}
lunghezza media: 573 caratteri


## 9. In una group chat nessuno parla con nessuno

Vale la pena guardare la **forma** di quello che abbiamo costruito, perche' non e' quella che si immagina.

La topologia di una group chat e' una **stella**: ogni partecipante ha archi solo verso l'orchestratore, e archi **tra partecipanti non esistono, per costruzione**. Il Fan non manda niente al Critico. Mette la sua battuta nella **conversazione condivisa**, e il Critico la legge da li'.

Non e' una sottigliezza da diagramma, ha due conseguenze che pagherai:

| | group chat (oggi) | grafo disegnato (EP 5) |
| --- | --- | --- |
| topologia | stella attorno all'orchestratore | archi che decidi tu |
| chi vede cosa | **tutti vedono tutto** | **decidi tu** cosa riceve ogni nodo |
| costo | ogni turno rispedisce l'intera conversazione | passa solo cio' che serve |

Il confronto tra le due puntate quindi non e' solo "chi decide il percorso": e' anche **"chi vede cosa"**.

Una conseguenza pratica: **il messaggio iniziale lo leggono tutti**, non solo il Moderatore. Per questo in sez. 7 l'abbiamo scritto come una **scheda** (`Film: ... Motion: ... Durata: ...`) e non come un ordine rivolto al gruppo, e per questo le istruzioni di Fan e Critico dicono che ognuno parla solo per se'. Un "fate 2 round" al plurale, ciascuno dei due lo prenderebbe come un compito suo.


## 10. Dal notebook al progetto

Il cast e' arrivato a quattro agenti, e un notebook comincia a stare stretto. Da qui in poi il Debate Club vive come **progetto**, in `progetto/debate_club/`: lo stesso codice di queste celle, messo come si mette in un progetto vero.

Non e' codice nuovo. E' **questo** codice, che ha smesso di stare in una cella:

| qui nel notebook | nel progetto |
| --- | --- |
| `Agent(name="Fan", ...)` | `agenti/fan.py` |
| `ISTRUZIONI_FAN` come stringa | `istruzioni/fan.md`, fuori dal codice |
| `GroupChatBuilder(...)` | `orchestrazione.py` |
| `limite_round_da_messaggio` | `orchestrazione.py`, con la sua docstring |
| `DossierFilm` (importato) | `modelli.py`, dove era gia' |
| questa cella che lanci col play | `cli.py`, che lanci con un comando |

Quattro scelte che vale la pena guardare, perche' non sono ovvie:

- **i contratti stanno in un posto solo** (`modelli.py`): quando piu' agenti si scambiano dati, l'errore facile e' che ognuno abbia la sua idea della forma;
- **Fan e Critico non hanno strumenti**: sembra una limitazione, ed e' il vincolo che in EP 6 permette al Giudice di dire se un numero e' inventato;
- **accanto a ogni tool c'e' la funzione pura**: cercare la filmografia non e' una decisione, e' un parsing, e non ha senso pagarla al modello;
- **il dossier e' verificato, non solo tipizzato**: l'output strutturato garantisce la *forma*, non la *verita'*. I dati li estrae il codice dalla pagina Wikipedia del film, e ogni citazione deve comparire parola per parola nel testo da cui viene, altrimenti esce.

### E ora guardalo muoversi

Dal terminale, nella cartella del progetto:

```
uv run debate-club dibatti "M. Night Shyamalan" --film 6
uv run debate-club devui
```

Il secondo comando apre una pagina locale dove trovi il Debate Club e i quattro agenti. Premi Run e guarda la timeline riempirsi: ad animarsi e' il **run** (testo in streaming, eventi, tracce), il disegno del grafo resta fermo.

Il form chiede **sei campi**, perche' DevUI lo genera dallo schema interno di `Message`: te ne servono due, `role` = `user` e `contents` = la scheda della sessione, scritta come scheda e non come ordine perche' la leggono tutti i partecipanti (sez. 9):

```
Film: The Village (2004), regia di M. Night Shyamalan.
Motion: capolavoro incompreso o semplicemente sopravvalutato?
Durata: 2 round a testa.
```

 Gli altri quattro si lasciano vuoti, **`author_name` compreso**, perche' e' proprio dall'assenza di un autore che la funzione dei round riconosce il tuo messaggio.

**Un dibattito per ogni avvio**: finito un dibattito, "Run Again" si chiude subito, perche' la group chat di MAF non azzera la conversazione tra un run e l'altro e DevUI riusa lo stesso workflow. Per il successivo: Ctrl+C e rilancia.

(DevUI e' un server locale: da qui dentro non si lancia, e su Colab non lo raggiungeresti.)

**Il tuo turno**: cambia la motion. "Il finale e' un colpo di genio o un imbroglio?" porta il dibattito da un'altra parte, con lo stesso dossier e lo stesso cast.

**In EP 5** lo stesso Debate Club diventa un **grafo disegnato da noi**: stessi agenti, percorso deciso a tavolino, e un punto in cui il sistema si **ferma e aspetta una persona**.